# Практикум 10. Дисбаланс классов

Сопровождает интерактивную лабораторию №10 («Дисбаланс классов»). Тема курса: несбалансированные выборки, ресэмплинг и веса классов

Ноутбук самодостаточен: первая ячейка создаёт файл `common.py` с учебным корпусом и вспомогательными функциями. Запускайте ячейки сверху вниз. Корпус учебный (шаблонный), для реальной задачи подставьте свой CSV со столбцами `text` и `label`.

In [ ]:
%%writefile common.py
"""Общие функции для практикумов по классификации и кластеризации текстов."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "спорт": ("матч команда гол игрок тренер чемпионат турнир победа поражение сезон стадион болельщик судья счёт "
              "защитник нападающий вратарь кубок лига финал трансфер клуб удар пенальти соревнование спортсмен "
              "рекорд медаль олимпиада тренировка ничья").split(),
    "политика": ("закон депутат выборы партия голосование президент министр правительство парламент реформа бюджет "
                 "санкции кандидат коалиция министерство указ поправка законопроект оппозиция референдум дипломатия "
                 "мэр губернатор конституция заседание комиссия переговоры саммит избиратель мандат полномочия").split(),
    "кулинария": ("суп рецепт соль мясо готовить духовка овощи сковорода кастрюля тесто сахар специи варить жарить "
                  "запекать соус бульон салат повар блюдо ингредиент масло мука яйцо молоко чеснок лук перец "
                  "картофель десерт выпечка начинка маринад").split(),
    "технологии": ("процессор программа приложение смартфон нейросеть алгоритм данные сервер обновление разработчик "
                   "компьютер память экран камера браузер сеть интернет облако код платформа пользователь устройство "
                   "чип батарея ноутбук софт видеокарта система интерфейс шифрование").split(),
}
NEUTRAL = ("сегодня очень новый говорят после который также можно было стал сообщили вчера недавно ещё всё "
           "несколько важный первый").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """Учебный корпус из коротких 'документов' четырёх тем.
    noise: доля слов, взятых не из своей темы (нейтральные или чужие). Чем больше noise, тем труднее задача."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """Проекция разреженной матрицы TF-IDF на плоскость (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Печатает слова с наибольшими весами для каждого класса (линейные модели)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))

In [ ]:
from common import *
big = make_corpus(n_per_class=300, noise=0.55, seed=1)
rare = "политика"
imb = pd.concat([big[big.label != rare], big[big.label == rare].sample(15, random_state=0)]).sample(frac=1, random_state=0)
imb["y"] = (imb.label == rare).astype(int)
print(imb.y.value_counts().to_dict(), "доля редкого класса:", round(imb.y.mean(), 3))

## 1. Базовая модель и парадокс точности

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.utils import resample

Xtr_t, Xte_t, ytr, yte = train_test_split(imb.text, imb.y, test_size=0.3, stratify=imb.y, random_state=0)
vec = TfidfVectorizer().fit(Xtr_t)
Xtr, Xte = vec.transform(Xtr_t), vec.transform(Xte_t)
def report(name, model, Xt=Xtr, yt=ytr):
    model.fit(Xt, yt); p = model.predict(Xte)
    return dict(модель=name, accuracy=accuracy_score(yte, p), precision=precision_score(yte, p, zero_division=0),
                recall=recall_score(yte, p), F1=f1_score(yte, p))
print("Всегда «не редкий»: accuracy =", round(1 - yte.mean(), 3))
rows = [report("без обработки", LogisticRegression(max_iter=1000))]
rows.append(report("веса классов", LogisticRegression(max_iter=1000, class_weight="balanced")))
pd.DataFrame(rows).round(3)

## 2. Ресэмплинг (только на обучающей выборке!)

In [ ]:
import scipy.sparse as sp
tr = pd.DataFrame({"t": ytr.index, "y": ytr.values})
pos, neg = tr[tr.y == 1], tr[tr.y == 0]
over = pd.concat([neg, resample(pos, n_samples=len(neg), random_state=0)])
under = pd.concat([resample(neg, n_samples=len(pos), replace=False, random_state=0), pos])
def fit_subset(sub, name):
    Xs = vec.transform(imb.loc[sub.t, "text"]); return report(name, LogisticRegression(max_iter=1000), Xs, sub.y.values)
rows += [fit_subset(over, "oversampling"), fit_subset(under, "undersampling")]
try:
    from imblearn.over_sampling import SMOTE
    Xs, ys = SMOTE(random_state=0, k_neighbors=3).fit_resample(Xtr, ytr)
    rows.append(report("SMOTE", LogisticRegression(max_iter=1000), Xs, ys))
except ImportError:
    print("Для SMOTE: !pip install imbalanced-learn")
pd.DataFrame(rows).round(3)

## 3. Порог решения и PR-кривая

In [ ]:
from sklearn.metrics import precision_recall_curve
m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
pr = m.predict_proba(Xte)[:, 1]
P, R, T = precision_recall_curve(yte, pr)
f1 = 2 * P * R / (P + R + 1e-9)
best = T[np.argmax(f1[:-1])]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(R, P); ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision"); ax[0].set_title("PR-кривая")
ax[1].plot(T, P[:-1], label="precision"); ax[1].plot(T, R[:-1], label="recall"); ax[1].axvline(best, color="gray", ls="--", label=f"лучший порог {best:.2f}")
ax[1].set_xlabel("порог"); ax[1].legend(); plt.show()
print(classification_report(yte, (pr >= best).astype(int), zero_division=0))

## Задания

1. Выведите долю редкого класса в 20%, 5%, 1%. Как ведут себя разные методы?
2. Почему ресэмплинг нельзя делать до разбиения на train и test?
3. Подберите порог по требованию «recall не ниже 0.9». Какой precision получается?